# HW2: Word2Vector
## Packages

In [2]:
import json
import string
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from collections import Counter
import jieba
import nltk
from scipy.spatial import distance
from tqdm import tqdm
import random
from sklearn.manifold import TSNE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Pre-produce
###  1.1 Create Vocabulary Table

In [3]:
# load txt text from ./data 
def load_data(path, version='zh'):
    with open(path+version+'.txt', 'r', encoding='utf-8') as f:
        text = f.read().replace('\n', '<EOS>')
    return text

# split text to words
def preprocess_text(text, zh_stop_path=None, language='zh'):
    if language == 'zh':
        # load stopwords(from https://github.com/goto456/stopwords)
        with open(zh_stop_path, 'r', encoding='utf-8') as f:
            stops = f.readlines()
            stop_words = set(stop.replace('\n','') for stop in stops)
        
        # remove stopwords and split words
        corpus = [word for word in text.replace('<', ' ').replace('>', ' ').split(' ') if word not in stop_words]
        return corpus
    else:
        # nltk.download('punkt_tab')
        # split
        # words = nltk.word_tokenize(text)
        words = text.replace('<', ' ').replace('>', ' ').split(' ')
        # remove stopwords and punctuations
        stop_words = set(nltk.corpus.stopwords.words('english'))
        tmp_words = [word.strip(string.punctuation) for word in words if word.strip(string.punctuation) not in string.whitespace]
        corpus = [word for word in tmp_words if word not in stop_words]
        return corpus

# build vocab
def build_vocab(corpus):
    # deduplicate and count frequency of words
    counter = Counter(corpus)
    # add 'UNK' for each word not in vocab
    vocab = {'UNK':0}
    # vocab is a dict{word1:num1, word2:num2, ...}, num1 > num2 > ...
    vocab.update({word:i+1 for i, (word, count) in enumerate(counter.most_common())})
    return vocab

### 1.2 Create Training Data

In [3]:
# create context-center word pairs([context_nums], center_num)
def create_training_data(corpus, vocab, window_size=5):
    train_dataset = []
    str_sorpus = ' '.join(corpus)
    sentences = str_sorpus.split('EOS') 
    sentence_list = []
    for sentence in sentences:
        sentence_list.append(sentence.split(' '))
    for sentence in sentence_list:
        for center_idx in range(window_size, len(sentence)-window_size):
            # get context words and center word
            context_words = sentence[center_idx-window_size:center_idx] + sentence[center_idx+1:center_idx+window_size+1]
            center_word = sentence[center_idx]
            # get idxs of context words and center word
            context_idxs = []
            for word in context_words:
                if vocab.get(word) is None:
                    context_idxs.append(0)
                else:
                    context_idxs.append(vocab.get(word))
                    
            if vocab.get(center_word) is None:
                center_idx = 0
            else:
                center_idx = vocab.get(center_word)
            # return pairs
            train_dataset.append((context_idxs, center_idx))
    return train_dataset

##  2. CBOW Model
### Use CBOW model to do this task.

In [4]:
# CBOW model
class CBOW(nn.Module):
    def __init__(self, vocab_size, embedding_size):
        super(CBOW, self).__init__()
        self.embedding_size = embedding_size
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.fc1 = nn.Linear(embedding_size, vocab_size)
        
    def forward(self, inputs):
        # pytorch.nn.Embedding transform inputs(a list of tokens' ints) into an embedding vector, 
        # just like using a one-hot vector to select an embedding vector from lookup table.
        embeddings = self.embedding(inputs)
        embeddings_mean = embeddings.mean(dim=1)
        out = self.fc1(embeddings_mean)
        log_probs = torch.log_softmax(out, dim=1)
        return log_probs

## 3. Optimizer & Loss Function


In [5]:
# optimizer & loss function
def get_optim_and_loss(model, optimizer="SGD", criterion="cross"):
    # optimizer
    if optimizer == "SGD":
        optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
    elif optimizer == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    # loss
    if criterion == "cross":
        criterion = nn.CrossEntropyLoss()
    else:
        criterion = nn.MSELoss()
    return optimizer, criterion

##  4. Train

In [1]:
# train
def train(model, data, optimizer, criterion, epochs, lossv, batch_size=64):
    for epoch in range(epochs):
        random.seed(epoch)
        random.shuffle(data)
        total_loss = 0.0
        for i in tqdm(range(0, len(data), batch_size)):
            batch = data[i:i + batch_size]
            # get contexts and centers
            context = torch.tensor([context_idxs for context_idxs, _ in batch]).to(device)
            center = torch.tensor([center_idx for _, center_idx in batch]).to(device)
            optimizer.zero_grad()
            log_probs = model(context)
            # for nn.CrossEntropyLoss:
            # shape of model output logits(log_probs) is (N, C). N is batch size and C is number of class.
            # shape of target(center_num) is (N,), each elem inside is a vector index of sample's true class
            loss = criterion(log_probs, center)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        lossv.append(total_loss/len(data))
        print(f'epoch {epoch}, loss: {total_loss/len(data)}')
    print('Training finished')

## 5. Save Word Vectors

In [7]:
# save word vectors
def save_word_vectors(model, vocab, save_vec_path, version="zh"):
    word_vectors = {}
    # get embbing weights
    embbeding_weights = model.embedding.weight.data
    for word, i in vocab.items():
        word_vectors[word] = embbeding_weights[i].cpu().numpy().tolist()
    
    # save as json format
    with open(save_vec_path+version+".vec", 'w', encoding='utf-8') as f:
        json.dump(word_vectors, f, ensure_ascii=False, indent=4)

# load word vectors
def load_word_vectors(save_vec_path, version="zh"):
    with open(save_vec_path+version+".vec", 'r', encoding='utf-8') as f:
        word_vectors = json.load(f)
    return word_vectors
        
# save model
def save_model(model, save_model_path, version="zh"):
    torch.save(model.state_dict(), save_model_path+version+"_cbow.pth")

# load model
def load_model(model, model_path, version="zh"):
    return model.load_state_dict(torch.load(model_path+version+"_cbow.pth"))

# 6. Case: Cosine Similarity

In [25]:
# find similar words
def find_similar_words(word, word_vectors, top_n=5):
    similarities = {}
    word_vec = word_vectors.get(word)
    for target_word, target_vec in word_vectors.items():
        if target_word != word:
            similarity = distance.cosine(target_vec, word_vec)
            similarities[target_word] = similarity
            
    top_similarities = sorted(similarities.items(), key=lambda x: x[1], reverse=True)[:top_n]
    return top_similarities